# 01 · Constructs and data checks

**Run this notebook first.** It reads the only two raw inputs (`data/task_master.csv`, `data/final_aei_panel.csv`), runs the Appendix B data checks, and writes every intermediate construct to `outputs/0_constructs/`. Notebooks 02–04 read those CSVs back instead of recomputing them, so every figure and table is built from the same numbers.

| Construct (thesis section) | File in `outputs/0_constructs/` |
|---|---|
| Task-level observed exposure r̃ₜ, αₜ, WorkUsageₜ (3.2.1) | `task_exposure.csv` |
| Subprocess / process observed exposure Rₒ,ₛₚ, Rₒ,ₚ (3.2.1) | `subprocess_exposure.csv`, `process_exposure.csv` |
| Task dispersion contributions (3.2.2) | `task_dispersion.csv` |
| Subprocess / process concentration (3.2.2) | `subprocess_concentration.csv`, `process_concentration.csv` |
| Task success by platform and pooled (3.2.3) | `task_success.csv` |
| Task coverage and effective exposure (3.2.3) | `effective_exposure.csv` |
| R2R-relative cost / input / output indices (3.1.4) | `r2r_indices.csv` |

All shared logic lives in `r2r_common.py`; this notebook only calls it and saves the results.

In [1]:
import numpy as np
import pandas as pd
import r2r_common as rc

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
tm, panel = rc.load_data()

[load] task_master: 212 rows, 91 in scope ({'Master Data Management': 8, 'Record': 31, 'Close & Consolidation': 5, 'Report': 13, 'Analysis & Business Insight': 12, 'Planning & Budgeting': 22})
[load] panel: 9,522 rows, releases [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)], 50 tasks; rows whose task is not in scope: 0


## 1. Data checks (Appendix B)

* **Table B1** — `task_id` uniqueness is enforced in `load_data()` (it raises if any id repeats); the cell below also lists the task master's columns.
* **Appendix B.2** — the coverage table shows which dimension/variable combinations each release publishes, which documents why Release 6 is excluded.

In [2]:
tm_all = pd.read_csv(rc.DATA_DIR / rc.TASK_MASTER_FILE, dtype={"task_id": str})
print("task_id unique across all", len(tm_all), "rows:", not tm_all["task_id"].duplicated().any())
print("task_master columns:", list(tm_all.columns))
print("frequency-like columns:", [c for c in tm_all.columns if "freq" in c.lower()])
print("in-scope tasks by subprocess:\n", tm["subprocess"].value_counts().reindex(rc.SUBPROCESS_ORDER))

task_id unique across all 212 rows: True
task_master columns: ['task_id', 'task_statement', 'subprocess', 'occupation_id', 'occupation_title', 'wage_hourly_mean', 'wage_annual_mean', 'ridge_task_years', 'task_time_fraction', 'frequency_occ_per_year', 'beta']
frequency-like columns: ['frequency_occ_per_year']
in-scope tasks by subprocess:
 subprocess
Master Data Management          8
Record                         31
Close & Consolidation           5
Report                         13
Analysis & Business Insight    12
Planning & Budgeting           22
Name: count, dtype: int64


In [3]:
coverage = (panel.groupby(["dimension_n", "variable_n", "platform_n", "release"])["task_id"]
            .nunique().unstack("release").fillna(0).astype(int))
rc.save_csv(coverage.reset_index(), "constructs", "panel_coverage_by_release.csv")

# Facets the analysis needs, and whether Release 6 publishes them
needed = [("usage", "count"), ("usage", "pct"), ("use_case", "count"), ("collaboration", "count"),
          ("task_success", "pct"), ("human_only_time", "mean"), ("ai_autonomy", "mean"),
          ("prompt_tokens", "index"), ("cost", "index")]
check = []
for dim, var in needed:
    for plat in rc.PLATFORMS:
        row = {"dimension": dim, "variable": var, "platform": plat}
        for r in range(1, 7):
            m = (panel["dimension_n"] == dim) & (panel["variable_n"] == var) & (panel["platform_n"] == plat) & (panel["release"] == r)
            row[f"R{r}"] = panel.loc[m, "task_id"].nunique()
        check.append(row)
release_check = pd.DataFrame(check)
rc.save_csv(release_check, "constructs", "release_facet_check.csv")
release_check

[write] outputs/0_constructs/panel_coverage_by_release.csv
[write] outputs/0_constructs/release_facet_check.csv


,dimension,variable,platform,R1,R2,R3,R4,R5,R6
0,usage,count,claude_ai,0,0,28,35,38,0
1,usage,count,1p_api,0,0,22,26,28,0
2,usage,pct,claude_ai,36,39,28,35,38,0
3,usage,pct,1p_api,0,0,22,26,28,0
4,use_case,count,claude_ai,0,0,0,35,38,0
5,use_case,count,1p_api,0,0,0,26,28,0
6,collaboration,count,claude_ai,0,0,28,35,38,0
7,collaboration,count,1p_api,0,0,22,26,28,0
8,task_success,pct,claude_ai,0,0,0,35,38,0
9,task_success,pct,1p_api,0,0,0,26,28,0


## 2. Observed exposure (Section 3.2.1)

Task-level r̃ₜ for Releases 4–5 under gate ON (WorkUsageₜ ≥ 100) and gate OFF, aggregated to subprocesses and to the pooled process with time-fraction weights.

In [4]:
task_exp, sub_exp, proc_exp = rc.build_exposure_tables(tm, panel)
rc.save_csv(task_exp, "constructs", "task_exposure.csv")
rc.save_csv(sub_exp, "constructs", "subprocess_exposure.csv")
rc.save_csv(proc_exp, "constructs", "process_exposure.csv")

# Values quoted in Section 4.1.2 (Figure 6 / Figure D1)
print(sub_exp.pivot_table(index="subprocess", columns=["release", "gate_on"], values="R_o")
      .reindex(rc.SUBPROCESS_ORDER).round(2))
print(proc_exp[["release", "gate_on", "R_process", "n_tasks_with_usage", "n_tasks_gate_passed"]].round(3))

[write] outputs/0_constructs/task_exposure.csv
[write] outputs/0_constructs/subprocess_exposure.csv
[write] outputs/0_constructs/process_exposure.csv
release                         4           5      
gate_on                     False True  False True 
subprocess                                         
Master Data Management       0.51  0.25  0.56  0.25
Record                       0.40  0.21  0.44  0.32
Close & Consolidation        0.44  0.44  0.40  0.40
Report                       0.15  0.06  0.21  0.06
Analysis & Business Insight  0.47  0.18  0.51  0.18
Planning & Budgeting         0.20  0.09  0.16  0.09
   release  gate_on  R_process  n_tasks_with_usage  n_tasks_gate_passed
0        4    False      0.326                  32                   32
1        4     True      0.162                  32                   16
2        5    False      0.351                  38                   38
3        5     True      0.197                  38                   18


## 3. Concentration (Section 3.2.2)

`concentration` (raw volume N, gate-invariant) is the variant used throughout the thesis; `concentration_gated` (N restricted to gate-passing tasks) is kept for sensitivity checks only.

In [5]:
task_disp = rc.compute_task_dispersion(task_exp, sub_exp, proc_exp)
sub_conc, proc_conc = rc.compute_concentration(task_disp)
rc.save_csv(task_disp, "constructs", "task_dispersion.csv")
rc.save_csv(sub_conc, "constructs", "subprocess_concentration.csv")
rc.save_csv(proc_conc, "constructs", "process_concentration.csv")

# Values quoted in Section 4.1.3 (Figure 8)
print(sub_conc.pivot_table(index="subprocess", columns=["gate_on", "release"], values="concentration")
      .reindex(rc.SUBPROCESS_ORDER).round(2))
print(proc_conc[["release", "gate_on", "R_process", "concentration", "concentration_gated"]].round(2))

[write] outputs/0_constructs/task_dispersion.csv
[write] outputs/0_constructs/subprocess_concentration.csv
[write] outputs/0_constructs/process_concentration.csv
gate_on                     False       True       
release                         4     5     4     5
subprocess                                         
Master Data Management       1.13  1.04  1.09  1.10
Record                       1.58  1.38  1.09  1.46
Close & Consolidation        1.13  0.93  1.13  0.93
Report                       0.72  0.74  0.31  0.31
Analysis & Business Insight  1.54  1.42  1.03  1.03
Planning & Budgeting         0.66  0.49  0.35  0.37
   release  gate_on  R_process  concentration  concentration_gated
0        4    False       0.33           1.60                 1.60
1        4     True       0.16           1.03                 1.02
2        5    False       0.35           1.47                 1.47
3        5     True       0.20           1.21                 1.20


## 4. Task success and effective exposure (Section 3.2.3)

In [6]:
succ = []
for r in rc.EXPOSURE_RELEASES:
    s = rc.task_success_table(panel, r).reset_index()
    s["release"] = r
    succ.append(s)
task_success = pd.concat(succ, ignore_index=True)
task_success["subprocess"] = task_success["task_id"].map(tm.set_index("task_id")["subprocess"])
rc.save_csv(task_success, "constructs", "task_success.csv")

eff = rc.compute_effective_exposure_table(tm, panel)
rc.save_csv(eff, "constructs", "effective_exposure.csv")
eff.pivot_table(index="subprocess", columns="release",
                values=["task_coverage_pct", "effective_exposure_pct"]).reindex(["Entire Process"] + rc.SUBPROCESS_ORDER).round(1)

[write] outputs/0_constructs/task_success.csv
[write] outputs/0_constructs/effective_exposure.csv


effective_exposure_pct       task_coverage_pct      
release                                          4     5                 4     5
subprocess                                                                      
Entire Process                                23.4  25.7              42.9  45.1
Master Data Management                        21.9  37.7              62.5  75.0
Record                                        25.7  26.5              48.4  41.9
Close & Consolidation                         31.5  29.2              40.0  40.0
Report                                        10.5   9.3              15.4  30.8
Analysis & Business Insight                   30.4  35.9              58.3  58.3
Planning & Budgeting                          24.0  26.9              36.4  40.9

## 5. R2R-relative token and cost indices (Section 3.1.4)

The AEI cost, prompt-token (input) and completion-token (output) indices are economy-wide; each is divided by its unweighted mean over the in-scope tasks of the release so that the R2R average is 1.0.

In [7]:
idx = rc.build_r2r_indices(panel, tm)
rc.save_csv(idx, "constructs", "r2r_indices.csv")
print(idx.groupby("release")[["cost_index", "input_index", "output_index"]].agg(["mean", "count"]).round(3))

[write] outputs/0_constructs/r2r_indices.csv
        cost_index       input_index       output_index      
              mean count        mean count         mean count
release                                                      
4              1.0    26         1.0    26          1.0    26
5              1.0    28         1.0    28          1.0    28
